# FixMatch: confidence-thresholded pseudo-labels meet consistency

_Semi & Self-Supervised Learning_

**Guess a label from a weak view; if you are sure enough, train a strong view to match it.**

FixMatch (Sohn et al., 2020) is the method that beat the complicated ones by being simple. It learns from a pile of unlabeled images with one short rule.

       For each unlabeled image, make a weakly augmented copy (a small flip or shift). Ask the model what it is. If the model is very sure — its top probability clears a high bar $\tau$ (for example 0.95) — write down that top guess as a fake "hard" label, called a pseudo-label.

---

This notebook is a practice scaffold for the **FixMatch: confidence-thresholded pseudo-labels meet consistency** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn.functional as F

def fixmatch_step(model, x_lab, y_lab, x_weak, x_strong,
                  tau=0.95, lambda_u=1.0):
    """One FixMatch optimization step.
      x_lab    : [B,  C,H,W]  weakly-augmented LABELED images
      y_lab    : [B]          their true labels
      x_weak   : [muB,C,H,W]  WEAKLY-augmented unlabeled images
      x_strong : [muB,C,H,W]  STRONGLY-augmented copies of the SAME images
    Returns the scalar total loss (call .backward() on it).
    """
    # ---- supervised term: plain cross-entropy on the few real labels ----
    logits_lab = model(x_lab)
    loss_s = F.cross_entropy(logits_lab, y_lab)

    # ---- weak view: build the pseudo-labels (no gradient through this) ----
    with torch.no_grad():
        logits_weak = model(x_weak)
        q = F.softmax(logits_weak, dim=1)        # q_b = p(y | weak(x_b))
        max_q, pseudo = q.max(dim=1)             # confidence and argmax label
        mask = (max_q >= tau).float()            # 1{ max(q_b) >= tau }

    # ---- strong view: train it to match the hard pseudo-label ----
    logits_strong = model(x_strong)              # gradients flow here
    # per-example cross-entropy H(pseudo_b, p(y | strong(x_b)))
    ce = F.cross_entropy(logits_strong, pseudo, reduction="none")
    loss_u = (ce * mask).mean()                  # average over the muB images

    # ---- total loss = supervised + lambda_u * unlabeled ----
    loss = loss_s + lambda_u * loss_u
    return loss, mask.mean()                      # also report the kept fraction


# --- usage sketch (RandAugment provides the strong augmentation) ---
# for (x_lab, y_lab), (x_uw, x_us) in zip(labeled_loader, unlabeled_loader):
#     loss, kept = fixmatch_step(model, x_lab, y_lab, x_uw, x_us, tau=0.95)
#     optimizer.zero_grad(); loss.backward(); optimizer.step()
#     # 'kept' rises from ~0 early (nothing clears tau) toward ~1 as the model improves

## Visualize the data & results

_FixMatch's central knob is the confidence threshold tau. As a faithful sklearn proxy, how does a self-trained classifier's accuracy change as we sweep that threshold, versus a labels-only baseline?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier

digits = load_digits()                         # 1797 real 8x8 handwritten digits
X, y = digits.data / 16.0, digits.target        # pixels scaled to 0..1
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=1, stratify=y)

# keep only 40 labels (4 per class); mark the rest unlabeled with -1
rng = np.random.RandomState(0)
lab_idx = []
for c in range(10):
    ci = rng.permutation(np.where(y_tr == c)[0])
    lab_idx += list(ci[:4])
lab_idx = np.array(lab_idx)
y_semi = y_tr.copy()
unlab = np.ones(len(y_tr), dtype=bool); unlab[lab_idx] = False
y_semi[unlab] = -1                              # SelfTrainingClassifier ignores -1

def base():                                    # confident base estimator
    return LogisticRegression(C=1.0, max_iter=2000)

# labels-only baseline: train on the 40 labels alone
baseline = base().fit(X_tr[lab_idx], y_tr[lab_idx]).score(X_te, y_te)

# sweep the confidence threshold = FixMatch's tau
for t in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]:
    st = SelfTrainingClassifier(base(), threshold=t, max_iter=100)
    st.fit(X_tr, y_semi)                        # uses labels + pseudo-labels >= t
    acc = st.score(X_te, y_te)
    print("tau=%.2f  self-train=%.3f  baseline=%.3f" % (t, acc, baseline))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Unlabeled image, 4 classes, $\tau = 0.95$. Weak-view prediction $q_b = [0.96,\ 0.02,\ 0.01,\ 0.01]$. Is it kept, and what is the pseudo-label?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find $\max(q_b)$, the largest probability. — _That is the model's confidence in its single best guess on the easy view._
- Compare it to $\tau = 0.95$ via the indicator $\mathbb{1}(\max(q_b) \ge \tau)$. — _The image is used only if confidence clears the threshold; otherwise it contributes nothing._
- If kept, take $\hat{q}_b = \arg\max q_b$, the index of that largest entry. — _The pseudo-label is the single hard class, not the whole distribution._

**Answer:** $\max(q_b) = 0.96$. Since $0.96 \ge 0.95$, the indicator is 1, so the image is kept. The pseudo-label is $\hat{q}_b = \arg\max q_b = $ class 0 (the first entry). The strong-augmented view of this image is then trained, by cross-entropy, to predict class 0.

</details>

**Problem 2.** Why does FixMatch read the pseudo-label from the WEAK view but apply the loss on the STRONG view? What would break if you swapped them?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall which view the model predicts most reliably. — _Mild augmentation keeps the image close to its true form, so the confident guess is more likely correct._
- Recall what the consistency loss is trying to force. — _We want the prediction to survive heavy distortion, which only pushes the model to generalize if the target view is hard._

**Answer:** The weak view is the most trustworthy, so its confident guess makes the cleanest pseudo-label. The strong view is the hardest, so forcing it to match teaches the model to generalize through heavy distortion — that is the consistency [unl-consistency] payoff. If you swapped them, you would read the label off a badly distorted image (often wrong, polluting the target) and train on an easy image (no generalization pressure). Both effects are lost, and confirmation bias gets worse.

</details>